In [ ]:
# 若没装 fastkaggle 就安装并导入
# install fastkaggle if not available
try: import fastkaggle
except ModuleNotFoundError:
    !pip install -Uq fastkaggle

from fastkaggle import *

In [ ]:
# 设置竞赛、下载数据、导入 fastai、固定种子
comp = 'paddy-disease-classification'
path = setup_comp(comp, install='fastai "timm>=0.6.2.dev0"')
from fastai.vision.all import *
set_seed(42)

In [ ]:
# 小图目录（先把图片缩小来加速实验）
trn_path = Path('sml')

In [ ]:
# 把训练图片统一缩到最大 256 存到 sml/
resize_images(path/'train_images', dest=trn_path, max_size=256, recurse=True)

In [ ]:
# 用小图建 DataLoaders 并预览
dls = ImageDataLoaders.from_folder(trn_path, valid_pct=0.2, seed=42,
    item_tfms=Resize((256,192)))

dls.show_batch(max_n=3)

In [ ]:
# 定义 train：给定架构 / 裁剪 / 增强，建模型并微调
def train(arch, item, batch, epochs=5):
    dls = ImageDataLoaders.from_folder(trn_path, seed=42, valid_pct=0.2, item_tfms=item, batch_tfms=batch)
    learn = vision_learner(dls, arch, metrics=error_rate).to_fp16()
    learn.fine_tune(epochs, 0.01)
    return learn

In [ ]:
# 基线：resnet26d
learn = train('resnet26d', item=Resize(192),
              batch=aug_transforms(size=128, min_scale=0.75))

In [ ]:
# 换更强的架构 convnext_small
arch = 'convnext_small_in22k'

In [ ]:
# convnext + squish 裁剪
learn = train(arch, item=Resize(192, method='squish'),
              batch=aug_transforms(size=128, min_scale=0.75))

In [ ]:
# convnext + 默认裁剪
learn = train(arch, item=Resize(192),
              batch=aug_transforms(size=128, min_scale=0.75))

In [ ]:
# 试试 Pad（补零）裁剪方式
dls = ImageDataLoaders.from_folder(trn_path, valid_pct=0.2, seed=42,
    item_tfms=Resize(192, method=ResizeMethod.Pad, pad_mode=PadMode.Zeros))
dls.show_batch(max_n=3)

In [ ]:
# convnext + Pad 裁剪训练
learn = train(arch, item=Resize((256,192), method=ResizeMethod.Pad, pad_mode=PadMode.Zeros),
      batch=aug_transforms(size=(171,128), min_scale=0.75))

In [ ]:
# 取验证集做预测
valid = learn.dls.valid
preds,targs = learn.get_preds(dl=valid)

In [ ]:
# 验证集 error_rate
error_rate(preds, targs)

In [ ]:
# 看训练集增强效果
learn.dls.train.show_batch(max_n=6, unique=True)

In [ ]:
# 用 TTA（测试时增强）预测验证集
tta_preds,_ = learn.tta(dl=valid)

In [ ]:
# TTA 后的 error_rate（通常更低更好）
error_rate(tta_preds, targs)

In [ ]:
# 换回全尺寸训练图片路径
trn_path = path/'train_images'

In [ ]:
# 用大图、Pad、12 轮正式训练
learn = train(arch, epochs=12,
              item=Resize((480, 360), method=ResizeMethod.Pad, pad_mode=PadMode.Zeros),
              batch=aug_transforms(size=(256,192), min_scale=0.75))

In [ ]:
# 验证集 TTA 的 error_rate
tta_preds,targs = learn.tta(dl=learn.dls.valid)
error_rate(tta_preds, targs)

In [ ]:
# 取测试图片，包成 test dataloader
tst_files = get_image_files(path/'test_images').sorted()
tst_dl = learn.dls.test_dl(tst_files)

In [ ]:
# 对测试集做 TTA 预测
preds,_ = learn.tta(dl=tst_dl)

In [ ]:
# 取每行概率最大的类别索引
idxs = preds.argmax(dim=1)

In [ ]:
# 把索引映射回类别名
vocab = np.array(learn.dls.vocab)
results = pd.Series(vocab[idxs], name="idxs")

In [ ]:
# 生成提交文件并查看
ss = pd.read_csv(path/'sample_submission.csv')
ss['label'] = results
ss.to_csv('subm.csv', index=False)
!head subm.csv

In [ ]:
# 提交到 Kaggle
if not iskaggle:
    from kaggle import api
    api.competition_submit_cli('subm.csv', 'convnext small 256x192 12 epochs tta', comp)

In [ ]:
# 推送 notebook 到 Kaggle（Jeremy 自用）
# This is what I use to push my notebook from my home PC to Kaggle

if not iskaggle:
    push_notebook('jhoward', 'small-models-road-to-the-top-part-2',
                  title='Small models: Road to the Top, Part 2',
                  file='small-models-road-to-the-top-part-2.ipynb',
                  competition=comp, private=True, gpu=True)